In [73]:
import pandas as pd
import numpy as np
import pickle
from datetime import timedelta
import os

import math
import tensorflow as tf
import random

import keras
from keras.layers import LSTM, Dense

from keras.models import Sequential 
from keras.layers import Dense, Dropout
from keras.optimizers import Adam, RMSprop
from keras.callbacks import EarlyStopping 
from scikeras.wrappers import KerasRegressor
from keras.models import load_model

random.seed(123)
np.random.seed(123)
tf.random.set_seed(123)
import gc
import pickle



from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler,LabelEncoder
from sklearn.model_selection import train_test_split,KFold,GridSearchCV
from sklearn.metrics import  r2_score, root_mean_squared_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor

import plotly.express as px
import plotly.graph_objects as go



# Expert data and auxiliary functions

## KC

In [74]:

# Define the data dictionary with the expert data
data = {
    "Start": [
        "2023-04-10", "2023-04-17", "2023-04-24", "2023-05-01", "2023-05-08",
        "2023-05-15", "2023-05-22", "2023-05-29", "2023-06-05", "2023-06-12",
        "2023-06-19", "2023-06-26", "2023-07-03", "2023-07-10", "2023-07-17",
        "2023-07-24", "2023-07-31", "2023-08-07", "2023-08-14", "2023-08-21",
        "2023-08-28", "2023-09-04", "2023-09-11", "2023-09-18", "2023-09-25",
        "2023-10-02", "2023-10-09", "2023-10-16", "2023-10-23"
    ],
    "End": [
        "2023-04-16", "2023-04-23", "2023-04-30", "2023-05-07", "2023-05-14",
        "2023-05-21", "2023-05-28", "2023-06-04", "2023-06-11", "2023-06-18",
        "2023-06-25", "2023-07-02", "2023-07-09", "2023-07-16", "2023-07-23",
        "2023-07-30", "2023-08-06", "2023-08-13", "2023-08-20", "2023-08-27",
        "2023-09-03", "2023-09-10", "2023-09-17", "2023-09-24", "2023-10-01",
        "2023-10-08", "2023-10-15", "2023-10-22", "2023-10-29"
    ],
    "Kc": [
        0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.58, 0.62, 0.62, 0.68, 
        0.68, 0.68, 0.68, 0.68, 0.68, 0.68, 0.55, 0.55, 0.49, 0.49, 0.49, 0.46, 
        0.46, 0.39, 0.39, 0.36, 0.36
    ]
}

# Create a DataFrame from the dictionary
fechas_kc = pd.DataFrame(data)

# Convert Start and End columns to datetime format
fechas_kc['Start'] = pd.to_datetime(fechas_kc['Start'])
fechas_kc['End'] = pd.to_datetime(fechas_kc['End'])

# Remove any rows with missing values (though none are expected in this dataset)
fechas_kc.dropna(inplace=True)

# Display the DataFrame to be sure everything is OK
fechas_kc


,Start,End,Kc
0,2023-04-10,2023-04-16,0.55
1,2023-04-17,2023-04-23,0.55
2,2023-04-24,2023-04-30,0.55
3,2023-05-01,2023-05-07,0.55
4,2023-05-08,2023-05-14,0.55
5,2023-05-15,2023-05-21,0.55
6,2023-05-22,2023-05-28,0.55
7,2023-05-29,2023-06-04,0.55
8,2023-06-05,2023-06-11,0.58
9,2023-06-12,2023-06-18,0.62


## Aux Functions

In [75]:

def CVRMSE(y_pred, y_true):
    """
    Computes the Coefficient of Variation of the Root Mean Square Error (CVRMSE).
    
    Parameters:
    - y_pred: array-like, predicted values
    - y_true: array-like, actual values

    Returns:
    - CVRMSE in percentage (%)
    """
    y_pred, y_true = np.array(y_pred), np.array(y_true)
    
    # Compute RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Compute CVRMSE
    cvrmse = (rmse / np.mean(y_true)) * 100
    
    return cvrmse


In [76]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def calcular_metricas(y_real, y_pred):
    """
    Computes multiple regression metrics:
    - R² (coefficient of determination)
    - MAE (Mean Absolute Error)
    - RMSE (Root Mean Squared Error)
    - CVRMSE (Coefficient of Variation of RMSE)

    Parameters:
    - y_real: array-like, actual values
    - y_pred: array-like, predicted values

    Returns:
    - Tuple (r2, mae, rmse, cvrmse)
    """
    # Compute R² using sklearn's r2_score function
    r2 = np.corrcoef(y_real, y_pred)[0][1]**2

    # Compute Mean Absolute Error (MAE)
    mae = mean_absolute_error(y_real, y_pred)

    # Compute Root Mean Squared Error (RMSE)
    rmse = root_mean_squared_error(y_real, y_pred)  

    # Compute CVRMSE
    cvrmse = CVRMSE(y_pred, y_real)  # Corrected argument order

    return r2, mae, rmse, cvrmse


In [77]:

def getKc(x):
    """
    Retrieves the Kc value for a given date x.
    Returns NaN if the date is outside the defined periods.
    
    Parameters:
    - x: datetime-like, the date to search for in fechas_kc

    Returns:
    - Corresponding Kc value or NaN if not found.
    """
    # Filter rows where x is within the Start and End range
    match = fechas_kc.loc[(fechas_kc['Start'] <= x) & (x < (fechas_kc['End'] + timedelta(days=1))), 'Kc']
    
    # Return the found value or NaN if not found
    return match.iloc[0] if not match.empty else np.nan


In [78]:
def print_metrics(id, r2, mae, rmse, cvrmse):
    """
    Prints regression metrics in a structured box format.

    Parameters:
    - id: Identifier for the dataset or plot.
    - r2: R² score.
    - mae: Mean Absolute Error.
    - rmse: Root Mean Squared Error.
    - cvrmse: Coefficient of Variation of RMSE (as a percentage).
    """
    metrics = [
        f'Plot: {id}',
        f'R²: {r2:.4f}',
        f'MAE: {mae:.4f}',
        f'RMSE: {rmse:.4f}',
        f'CVRMSE: {cvrmse:.2f}%'
    ]
    
    max_length = max(len(line) for line in metrics)
    border = '=' * (max_length + 4)  # +4 for padding and borders

    print(border)
    for line in metrics:
        print(f'= {line.ljust(max_length)} =')
    print(border)


In [79]:

def lags(df, lags_dic):
    """
    Creates lagged features for specified columns in a DataFrame.
    
    Parameters:
    - df (pd.DataFrame): The original DataFrame.
    - lags_dic (dict): Dictionary where keys are column names, 
      and values are lists of lag values to create.
    
    Returns:
    - pd.DataFrame: DataFrame with new lagged columns.
    """
    df = df.copy()  # Prevent modifying the original DataFrame
    
    for col, lag_list in lags_dic.items():
        for lag in lag_list:
            df[f'{col}_lag{lag}'] = df[col].shift(lag)
    
    return df


# Load data 

In [80]:
lags_dic = {
    'HR35_55': range(1,12),
    'Riego': range(1,12),
    'PREC': range(1,12),
    'TMED':range(1,12),
    'HR':range(1,12),
    'RAD':range(1,12),
    'DPV':range(1,12),
    'VV':range(1,12),
    'ETO':range(1,12)
}

In [81]:

# Get all filenames from the directory
plots = os.listdir('./Datos/2023/')
all_plots = []

for p in plots:
    # Read the Excel file
    df_fp = pd.read_excel(f'./Datos/2023/{p}', usecols='E,F,J,N,R,W,Y:AE')
    df_fp['DateTime'] = pd.to_datetime(df_fp['DateTime'])
    df_fp['hour_sin'] = df_fp['DateTime'].apply(lambda x: math.sin((2 * math.pi * x.hour) / 24))
    df_fp.set_index('DateTime', inplace=True)

    # Resample data to hourly intervals
    df_fp = df_fp.resample('1h').mean()

    # Rename columns for clarity
    df_fp.columns = ['FP', 'HR5_25', 'HR35_55', 'HR65_85', 'Riego', 'TMED', 'PREC', 'HR', 'RAD', 'DPV', 'VV', 'ETO', 'Hour_sin']

    # Drop unnecessary humidity columns
    df_fp.drop(columns=['HR5_25', 'HR65_85'], inplace=True)

    # Filter the date range (June 1 - Sept 30, 2023)
    df_fp = df_fp[df_fp.index >= '2023-6-1'].copy()
    df_fp = df_fp[df_fp.index < '2023-10-1'].copy()
    # Apply Kc values using getKc()
    df_fp['Kc'] = df_fp.reset_index()['DateTime'].apply(getKc).values

    # Extract plot ID from filename
    df_fp['ID'] = p.split('_')[0]
    df_fp.dropna(inplace=True)
    df_fp['FP'] = df_fp['FP'] * 0.1
  
    # Special filter for files containing '4.2' because sensor malfunction
    if '4.2' in p.split('_')[0]:
        df_fp = df_fp[df_fp.index >= '2023-07-26 12:00:00']

    
    all_plots.append(df_fp)





In [82]:

# Concatenate all DataFrames into a single dataset
df_all_plots = pd.concat(all_plots, ignore_index=True)

# Ensure only numeric columns are selected (excluding FP)
numerical_features = df_all_plots.select_dtypes(include=[np.number]).columns.drop('FP', errors='ignore')

scaler_x = MinMaxScaler().fit(df_all_plots[numerical_features])  # Scale independent features
scaler_y = MinMaxScaler().fit(df_all_plots[['FP']])  # Scale target variable


### Correlacion Cruzada


In [115]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def cross_correlation_with_lag(group, max_lag=24):
    """
    Calcula la correlación cruzada entre 'FP' y otras variables numéricas en un grupo (un árbol),
    considerando un rango de lags de -max_lag a +max_lag.
    """
    numeric_cols = group.select_dtypes(include=['number']).columns
    numeric_cols = numeric_cols.drop('FP', errors='ignore')  # Excluimos FP de sí misma

    results = []
    
    for col in numeric_cols:
        lags = range(-max_lag, max_lag + 1)  # Lags desde -24 hasta +24
        corrs = [group['FP'].shift(lag).corr(group[col]) for lag in lags]
        
        df_corrs = pd.DataFrame({'Lag': lags, 'Correlation': corrs, 'Variable': col})
        df_corrs['ID'] = group['ID'].iloc[0]  # Agregar el ID del árbol
        results.append(df_corrs)

    return pd.concat(results, ignore_index=True)

# Agrupar por árbol y calcular la correlación cruzada
df_all_plots_rename = df_all_plots.copy().rename(columns={
    'ETO': 'ETo',
    'TMED': 'T',
    'HR': 'Rh',
    'PREC': 'P',
    'RAD': 'Rs',
    'VV': 'U',
    'DPV': 'VPD',
    'HR35_55': 'SWC',
    'Riego': 'IR'
})
correlations_per_tree = df_all_plots_rename.groupby('ID').apply(cross_correlation_with_lag).reset_index(drop=True)

mean_correlations = correlations_per_tree.groupby(['Lag', 'Variable'])['Correlation'].mean().reset_index()


fig = go.Figure()
for variable in mean_correlations['Variable'].unique():
    subset = mean_correlations[mean_correlations['Variable'] == variable]
    fig.add_trace(go.Scatter(x=subset['Lag'], y=subset['Correlation'], mode='lines', name=variable))
      
fig.update_layout(
    xaxis_title="Lag",
    yaxis_title="Correlation",
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True),
    template="plotly_white",
    hovermode="x unified"
)

# Mostrar la gráfica
fig.show()



C:\Users\ricar\AppData\Local\Temp\ipykernel_4140\3894102468.py:39: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [84]:

train = pd.DataFrame(columns=all_plots[0].columns)
test = pd.DataFrame(columns=all_plots[0].columns)


train_list = []
test_list = []

# Process each plot separately
for p in all_plots:
    split_index = int(len(p) * 0.7)  # 70% train, 30% test

    # Append to lists instead of repeated concatenation
    train_list.append(p.iloc[:split_index, :])
    test_list.append(p.iloc[split_index:, :])

    print(f"Plot {p['ID'].iloc[0]}: total {len(p)}, train {split_index}, test {len(p) - split_index}")

# Concatenate all training and testing datasets at once
train = pd.concat(train_list, ignore_index=True)
test = pd.concat(test_list, ignore_index=True)



Plot T1.1.: total 1719, train 1203, test 516
Plot T1.2.: total 2928, train 2049, test 879
Plot T2.1.: total 2928, train 2049, test 879
Plot T2.2.: total 2928, train 2049, test 879
Plot T3.1.: total 2928, train 2049, test 879
Plot T4.1.: total 2928, train 2049, test 879
Plot T4.2.: total 1596, train 1117, test 479


### Encode the tree

In [85]:



# Combine IDs from train & test to ensure consistent encoding
encoder = LabelEncoder()
encoder.fit(pd.concat([train['ID'], test['ID']], ignore_index=True))

# Apply the same encoding to both sets
train['ID_encoded'] = encoder.transform(train['ID'])
test['ID_encoded'] = encoder.transform(test['ID'])


In [86]:
train_scaled = train.copy()

# Only numeric  are transformed
train_scaled[numerical_features] = scaler_x.transform(train[numerical_features].drop(columns=['ID','ID_encoded'], errors='ignore'))
train_scaled['FP'] = scaler_y.transform(train[['FP']])

# Generate lagged features for each unique ID
plots = []
for arbol in train_scaled['ID'].unique():
    data = train_scaled[train_scaled['ID'] == arbol]  
    data = lags(data, lags_dic)  
    plots.append(data)

train_scaled = pd.concat(plots, ignore_index=False)
train_scaled.dropna(inplace=True)

C:\Users\ricar\AppData\Local\Temp\ipykernel_4140\93538328.py:17: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\ricar\AppData\Local\Temp\ipykernel_4140\93538328.py:17: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\ricar\AppData\Local\Temp\ipykernel_4140\93538328.py:17: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame,

In [87]:

# Create a copy to avoid modifying the original test set
test_scaled = test.copy()

# Ensure only numerical features are transformed
test_scaled[numerical_features] = scaler_x.transform(test[numerical_features].drop(columns=['ID','ID_encoded'], errors='ignore'))

plots = []
for arbol in test_scaled['ID'].unique():
    data = test_scaled[test_scaled['ID'] == arbol]  
    data = lags(data, lags_dic)  
    plots.append(data)

test_scaled = pd.concat(plots, ignore_index=False)
test_scaled.dropna(inplace=True)

C:\Users\ricar\AppData\Local\Temp\ipykernel_4140\93538328.py:17: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\ricar\AppData\Local\Temp\ipykernel_4140\93538328.py:17: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\ricar\AppData\Local\Temp\ipykernel_4140\93538328.py:17: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame,

# Models

In [88]:
train_scaled.loc[:, ~train_scaled.columns.str.contains('_lag')].drop(columns=['FP','ID', 'ID_encoded']).columns

Index(['HR35_55', 'Riego', 'TMED', 'PREC', 'HR', 'RAD', 'DPV', 'VV', 'ETO',
       'Hour_sin', 'Kc'],
      dtype='object')

In [89]:
metricas_df = pd.DataFrame(columns=['Modelo', 'Arbol','R2', 'MAE', 'CVRMSE', 'RMSE'])

## RF

In [90]:
# Define Features & Target
X_train_rf = train_scaled.drop(columns=['FP', 'ID', 'ID_encoded'])
y_train_rf = train_scaled['FP']

param_grid = {
    'n_estimators': [100, 300, 500],  
    'max_depth': [10, 15, 20],  
}

# Initialize RandomForestRegressor
rf = RandomForestRegressor(random_state=123)

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,  
    n_jobs=-1, 
)

# Fit the Model
grid_search.fit(X_train_rf, y_train_rf)

# Get Best Parameters & Model
best_params = grid_search.best_params_
best_rf_model = grid_search.best_estimator_

print("Best parameters found: ", best_params)

Best parameters found:  {'max_depth': 15, 'n_estimators': 300}


In [111]:
for id in test_scaled['ID'].unique():
    # Select data for the current ID
    df_plot = test_scaled[test_scaled['ID'] == id]
    print(f'{id} {df_plot.shape}')

T1.1. (505, 113)
T1.2. (868, 113)
T2.1. (868, 113)
T2.2. (868, 113)
T3.1. (868, 113)
T4.1. (868, 113)
T4.2. (468, 113)


In [ ]:


for id in test_scaled['ID'].unique():
    # Select data for the current ID
    df_plot = test_scaled[test_scaled['ID'] == id]
    print(f'{id} {df_plot.shape}')

    X_test = df_plot.drop(columns=['FP', 'ID', 'ID_encoded'])
    y_test = df_plot['FP']


    y_pred = best_rf_model.predict(X_test)
    y_pred = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()  # Ensure correct shape

    # Compute evaluation metrics
    r2, mae, rmse, cvrmse = calcular_metricas(y_test, y_pred)
    print_metrics(id, r2, mae, rmse, cvrmse)

    # Plot predictions 
    fig_individual = go.Figure()
    fig_individual.add_trace(go.Scatter(x=df_plot.index, y=y_test, name='Real', mode='lines'))
    fig_individual.add_trace(go.Scatter(x=df_plot.index, y=y_pred, name='Predicción', mode='lines'))
    fig_individual.update_layout(
        title=f'Predicciones vs Real para el Plot {id}',
        xaxis_title='Date',
        yaxis_title='TWP (MPA)'
    )
    fig_individual.show()


    nueva_linea = pd.DataFrame({
        'Modelo': ['RandomForest'],
        'Arbol': [id],
        'R2': [r2],
        'MAE': [mae],
        'RMSE': [rmse],  
        'CVRMSE': [cvrmse]
    })

    metricas_df = pd.concat([metricas_df, nueva_linea], ignore_index=True)





T1.1. (505, 113)
= Plot: T1.1.     =
= R²: 0.6452      =
= MAE: 0.2000     =
= RMSE: 0.2374    =
= CVRMSE: -21.15% =


T1.2. (868, 113)
= Plot: T1.2.     =
= R²: 0.5793      =
= MAE: 0.1858     =
= RMSE: 0.2322    =
= CVRMSE: -17.25% =


T2.1. (868, 113)
= Plot: T2.1.     =
= R²: 0.4957      =
= MAE: 0.2810     =
= RMSE: 0.3559    =
= CVRMSE: -23.52% =


T2.2. (868, 113)
= Plot: T2.2.     =
= R²: 0.5560      =
= MAE: 0.1708     =
= RMSE: 0.2014    =
= CVRMSE: -16.01% =


T3.1. (868, 113)
= Plot: T3.1.     =
= R²: 0.2916      =
= MAE: 0.1982     =
= RMSE: 0.2403    =
= CVRMSE: -21.63% =


T4.1. (868, 113)
= Plot: T4.1.     =
= R²: 0.7148      =
= MAE: 0.3125     =
= RMSE: 0.3742    =
= CVRMSE: -33.54% =


T4.2. (468, 113)
= Plot: T4.2.     =
= R²: 0.6951      =
= MAE: 0.1856     =
= RMSE: 0.2305    =
= CVRMSE: -21.58% =


In [92]:

# Define Feature Matrices & Target Variables
X_train = train_scaled.drop(columns=['FP', 'ID', 'ID_encoded'])
y_train = train_scaled['FP']
X_test = test_scaled.drop(columns=['FP', 'ID', 'ID_encoded'])
y_test = test_scaled['FP']



# Inverse Transform Target Variables (Original Scale)
y_train = scaler_y.inverse_transform(y_train.values.reshape(-1, 1)).ravel()

# Make Predictions & Inverse Transform
y_pred = best_rf_model.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).ravel()

y_pred_train = best_rf_model.predict(X_train)
y_pred_train = scaler_y.inverse_transform(y_pred_train.reshape(-1, 1)).ravel()

# Compute evaluation metrics
r2_train, mae_train, rmse_train, cvrmse_train = calcular_metricas(y_train, y_pred_train)
r2, mae, rmse, cvrmse = calcular_metricas(y_test, y_pred)

# Print Metrics
print(f'R² - Test: {r2:.4f} | Train: {r2_train:.4f}')
print(f'MAE - Test: {mae:.4f} | Train: {mae_train:.4f}')
print(f'RMSE - Test: {rmse:.2f}% | Train: {rmse_train:.2f}%')
print(f'CVRMSE - Test: {cvrmse:.4f} | Train: {cvrmse_train:.4f}')


R² - Test: 0.4800 | Train: 0.9773
MAE - Test: 0.2230 | Train: 0.0605
RMSE - Test: 0.28% | Train: 0.09%
CVRMSE - Test: -22.6426 | Train: -9.0019


In [93]:


nueva_linea = pd.DataFrame({'Modelo': 'RandomForest', 'Arbol': 'All','R2':[r2], 'MAE': [mae], 'CVRMSE': [cvrmse], 'RMSE': [rmse]})
metricas_df = pd.concat([metricas_df, nueva_linea],ignore_index=True)

## KNN

In [94]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV

# Define the features and target
X_train_knn = train_scaled.drop(columns=['FP', 'ID', 'ID_encoded'])
y_train_knn = train_scaled['FP']

# Define the parameter grid for GridSearchCV
param_grid = {
    'n_neighbors': [7, 9, 11],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

# Initialize the KNeighborsRegressor
knn = KNeighborsRegressor()

# Initialize GridSearchCV
grid_search_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=3,  
    n_jobs=-1, 
)

# Fit the model
grid_search_knn.fit(X_train_knn, y_train_knn)

# Get the best parameters and best model
best_params_knn = grid_search_knn.best_params_
best_knn_model = grid_search_knn.best_estimator_

print("Best parameters found: ", best_params_knn)

Best parameters found:  {'algorithm': 'auto', 'n_neighbors': 7, 'weights': 'distance'}


In [95]:


for id in test_scaled['ID'].unique():
    # Select data for the current ID
    df_plot = test_scaled[test_scaled['ID'] == id]


    X_test = df_plot.drop(columns=['FP', 'ID', 'ID_encoded'])
    y_test = df_plot['FP']


    y_pred = best_knn_model.predict(X_test)
    y_pred = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()  # Ensure correct shape

    # Compute evaluation metrics
    r2, mae, rmse, cvrmse = calcular_metricas(y_test, y_pred)
    print_metrics(id, r2, mae, rmse, cvrmse)

    # Plot predictions 
    fig_individual = go.Figure()
    fig_individual.add_trace(go.Scatter(x=df_plot.index, y=y_test, name='Real', mode='lines'))
    fig_individual.add_trace(go.Scatter(x=df_plot.index, y=y_pred, name='Predicción', mode='lines'))
    fig_individual.update_layout(
        title=f'Predicciones vs Real para el Plot {id}',
        xaxis_title='Date',
        yaxis_title='TWP (MPA)'
    )
    fig_individual.show()


    nueva_linea = pd.DataFrame({
        'Modelo': ['KNN'],
        'Arbol': [id],
        'R2': [r2],
        'MAE': [mae],
        'RMSE': [rmse],  
        'CVRMSE': [cvrmse]
    })

    metricas_df = pd.concat([metricas_df, nueva_linea], ignore_index=True)





= Plot: T1.1.     =
= R²: 0.5839      =
= MAE: 0.1773     =
= RMSE: 0.2188    =
= CVRMSE: -19.49% =


= Plot: T1.2.     =
= R²: 0.6224      =
= MAE: 0.2082     =
= RMSE: 0.2564    =
= CVRMSE: -19.05% =


= Plot: T2.1.     =
= R²: 0.5531      =
= MAE: 0.2548     =
= RMSE: 0.3189    =
= CVRMSE: -21.08% =


= Plot: T2.2.     =
= R²: 0.5673      =
= MAE: 0.1730     =
= RMSE: 0.2111    =
= CVRMSE: -16.78% =


= Plot: T3.1.     =
= R²: 0.5247      =
= MAE: 0.1873     =
= RMSE: 0.2331    =
= CVRMSE: -20.98% =


= Plot: T4.1.     =
= R²: 0.6305      =
= MAE: 0.3236     =
= RMSE: 0.3843    =
= CVRMSE: -34.45% =


= Plot: T4.2.     =
= R²: 0.7098      =
= MAE: 0.1498     =
= RMSE: 0.1809    =
= CVRMSE: -16.93% =


In [96]:

# Define Feature Matrices & Target Variables
X_train = train_scaled.drop(columns=['FP', 'ID', 'ID_encoded'])
y_train = train_scaled['FP']
X_test = test_scaled.drop(columns=['FP', 'ID', 'ID_encoded'])
y_test = test_scaled['FP']



# Inverse Transform Target Variables (Original Scale)
y_train = scaler_y.inverse_transform(y_train.values.reshape(-1, 1)).ravel()

# Make Predictions & Inverse Transform
y_pred = best_knn_model.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).ravel()

y_pred_train = best_rf_model.predict(X_train)
y_pred_train = scaler_y.inverse_transform(y_pred_train.reshape(-1, 1)).ravel()

# Compute evaluation metrics
r2_train, mae_train, rmse_train, cvrmse_train = calcular_metricas(y_train, y_pred_train)
r2, mae, rmse, cvrmse = calcular_metricas(y_test, y_pred)

# Print Metrics
print(f'R² - Test: {r2:.4f} | Train: {r2_train:.4f}')
print(f'MAE - Test: {mae:.4f} | Train: {mae_train:.4f}')
print(f'RMSE - Test: {rmse:.2f}% | Train: {rmse_train:.2f}%')
print(f'CVRMSE - Test: {cvrmse:.4f} | Train: {cvrmse_train:.4f}')


R² - Test: 0.5103 | Train: 0.9773
MAE - Test: 0.2174 | Train: 0.0605
RMSE - Test: 0.27% | Train: 0.09%
CVRMSE - Test: -22.1465 | Train: -9.0019


In [97]:


nueva_linea = pd.DataFrame({'Modelo': 'KNN', 'Arbol': 'All','R2':[r2], 'MAE': [mae],  'CVRMSE': [cvrmse], 'RMSE': [rmse]})
metricas_df = pd.concat([metricas_df, nueva_linea],ignore_index=True)

## SVR

In [98]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

# Define the features and target
X_train_svr = train_scaled.drop(columns=['FP', 'ID', 'ID_encoded'])
y_train_svr = train_scaled['FP']

# Define the parameter grid for GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 100],
    'epsilon': [0.01, 0.1, 0.2],
    'kernel': ['linear', 'poly', 'rbf']
}

# Initialize the SVR model
svr = SVR()

# Initialize GridSearchCV
grid_search_svr = GridSearchCV(estimator=svr, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)

# Fit the model
grid_search_svr.fit(X_train_svr, y_train_svr)

# Get the best parameters and best model
best_params_svr = grid_search_svr.best_params_
best_svr_model = grid_search_svr.best_estimator_

print("Best parameters found: ", best_params_svr)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best parameters found:  {'C': 10, 'epsilon': 0.1, 'kernel': 'poly'}


In [99]:


for id in test_scaled['ID'].unique():
    # Select data for the current ID
    df_plot = test_scaled[test_scaled['ID'] == id]


    X_test = df_plot.drop(columns=['FP', 'ID', 'ID_encoded'])
    y_test = df_plot['FP']


    y_pred = best_svr_model.predict(X_test)
    y_pred = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()  # Ensure correct shape

    # Compute evaluation metrics
    r2, mae, rmse, cvrmse = calcular_metricas(y_test, y_pred)
    print_metrics(id, r2, mae, rmse, cvrmse)

    # Plot predictions 
    fig_individual = go.Figure()
    fig_individual.add_trace(go.Scatter(x=df_plot.index, y=y_test, name='Real', mode='lines'))
    fig_individual.add_trace(go.Scatter(x=df_plot.index, y=y_pred, name='Predicción', mode='lines'))
    fig_individual.update_layout(
        title=f'Predicciones vs Real para el Plot {id}',
        xaxis_title='Date',
        yaxis_title='TWP (MPA)'
    )
    fig_individual.show()


    nueva_linea = pd.DataFrame({
        'Modelo': ['SVR'],
        'Arbol': [id],
        'R2': [r2],
        'MAE': [mae],
        'RMSE': [rmse],  
        'CVRMSE': [cvrmse]
    })

    metricas_df = pd.concat([metricas_df, nueva_linea], ignore_index=True)





= Plot: T1.1.     =
= R²: 0.3969      =
= MAE: 0.1825     =
= RMSE: 0.2338    =
= CVRMSE: -20.83% =


= Plot: T1.2.     =
= R²: 0.4242      =
= MAE: 0.2304     =
= RMSE: 0.2901    =
= CVRMSE: -21.55% =


= Plot: T2.1.     =
= R²: 0.4334      =
= MAE: 0.2674     =
= RMSE: 0.3625    =
= CVRMSE: -23.96% =


= Plot: T2.2.     =
= R²: 0.5139      =
= MAE: 0.1893     =
= RMSE: 0.2430    =
= CVRMSE: -19.31% =


= Plot: T3.1.     =
= R²: 0.4289      =
= MAE: 0.1955     =
= RMSE: 0.2431    =
= CVRMSE: -21.89% =


= Plot: T4.1.     =
= R²: 0.5318      =
= MAE: 0.3231     =
= RMSE: 0.3850    =
= CVRMSE: -34.51% =


= Plot: T4.2.     =
= R²: 0.6174      =
= MAE: 0.1551     =
= RMSE: 0.1968    =
= CVRMSE: -18.43% =


In [100]:

# Define Feature Matrices & Target Variables
X_train = train_scaled.drop(columns=['FP', 'ID', 'ID_encoded'])
y_train = train_scaled['FP']
X_test = test_scaled.drop(columns=['FP', 'ID', 'ID_encoded'])
y_test = test_scaled['FP']



# Inverse Transform Target Variables (Original Scale)
y_train = scaler_y.inverse_transform(y_train.values.reshape(-1, 1)).ravel()

# Make Predictions & Inverse Transform
y_pred = best_svr_model.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).ravel()

y_pred_train = best_rf_model.predict(X_train)
y_pred_train = scaler_y.inverse_transform(y_pred_train.reshape(-1, 1)).ravel()

# Compute evaluation metrics
r2_train, mae_train, rmse_train, cvrmse_train = calcular_metricas(y_train, y_pred_train)
r2, mae, rmse, cvrmse = calcular_metricas(y_test, y_pred)

# Print Metrics
print(f'R² - Test: {r2:.4f} | Train: {r2_train:.4f}')
print(f'MAE - Test: {mae:.4f} | Train: {mae_train:.4f}')
print(f'RMSE - Test: {rmse:.2f}% | Train: {rmse_train:.2f}%')
print(f'CVRMSE - Test: {cvrmse:.4f} | Train: {cvrmse_train:.4f}')


R² - Test: 0.4478 | Train: 0.9773
MAE - Test: 0.2280 | Train: 0.0605
RMSE - Test: 0.30% | Train: 0.09%
CVRMSE - Test: -23.8883 | Train: -9.0019


In [101]:
nueva_linea = pd.DataFrame({'Modelo': 'SVR', 'Arbol': 'All','R2':[r2], 'MAE': [mae], 'CVRMSE': [cvrmse], 'RMSE': [rmse]})
metricas_df = pd.concat([metricas_df, nueva_linea],ignore_index=True)

In [102]:
metricas_df.to_csv('./metricas_df.csv', index=False)
metricas_df

,Modelo,Arbol,R2,MAE,CVRMSE,RMSE
0,RandomForest,T1.1.,0.645202,0.199966,-21.152170,0.237440
1,RandomForest,T1.2.,0.579314,0.185806,-17.248445,0.232195
2,RandomForest,T2.1.,0.495692,0.281045,-23.521454,0.355892
3,RandomForest,T2.2.,0.556045,0.170810,-16.005980,0.201426
4,RandomForest,T3.1.,0.291609,0.198181,-21.634345,0.240296
5,RandomForest,T4.1.,0.714808,0.312523,-33.543781,0.374191
6,RandomForest,T4.2.,0.695079,0.185579,-21.579174,0.230501
7,RandomForest,All,0.480048,0.222965,-22.642632,0.280137
8,KNN,T1.1.,0.583859,0.177272,-19.492838,0.218813
9,KNN,T1.2.,0.622374,0.208215,-19.046607,0.256401


In [103]:
for id in test_scaled['ID'].unique():
    # Obtener el dataframe correspondiente al plot
    df_plot = test_scaled[test_scaled['ID'] == id]


    # Crear entradas y salidas
    X_test = df_plot.drop(columns=['FP', 'ID', 'ID_encoded'])
    y_test = df_plot['FP']
    

    # Realizar predicciones en el conjunto de prueba
    y_pred_rf = best_rf_model.predict(X_test)
    y_pred_rf = scaler_y.inverse_transform(y_pred_rf.reshape(-1, 1)).ravel()

    y_pred_knn = best_knn_model.predict(X_test)
    y_pred_knn = scaler_y.inverse_transform(y_pred_knn.reshape(-1, 1)).ravel()

    y_pred_svr = best_svr_model.predict(X_test)
    y_pred_svr= scaler_y.inverse_transform(y_pred_svr.reshape(-1, 1)).ravel()



    # Crear una figura para el plot individual
    fig_individual = go.Figure()
    fig_individual.add_trace(go.Scatter(x=df_plot.index, y=y_test, name='Real', mode='lines'))
    fig_individual.add_trace(go.Scatter(x=df_plot.index, y=y_pred_rf, name='RandomForest', mode='lines'))
    fig_individual.add_trace(go.Scatter(x=df_plot.index, y=y_pred_knn, name='KNN', mode='lines'))
    fig_individual.add_trace(go.Scatter(x=df_plot.index, y=y_pred_svr, name='SVR', mode='lines'))
    fig_individual.update_layout(title=f'Predicciones vs Real para el Plot {id}', xaxis_title='Date', yaxis_title='TWP(MPA)')
    fig_individual.show()
